# Part D: Food-order stream with RabbitMQ (run in Google Colab)

## 1. Install and start a local RabbitMQ broker

In [ ]:
!apt-get install -y -q rabbitmq-server > /dev/null 2>&1
!service rabbitmq-server start
!pip install -q pika

## 2. Get the scripts from GitHub

In [ ]:
!rm -rf freshcart-pipelines
!git clone -q https://github.com/Shristirajpoot/freshcart-pipelines.git
%cd freshcart-pipelines/stream
!ls

## 3. Start six drivers (private queue + binding pattern each)

In [ ]:
import subprocess, time
DRIVERS = {"Ravi": "bangalore.koramangala.*", "Priya": "bangalore.indiranagar.*", "Arjun": "bangalore.*.ride",
           "Meena": "bangalore.*.food", "Karthik": "chennai.tnagar.food", "Control": "bangalore.#"}
procs = {}
for name, pattern in DRIVERS.items():
    procs[name] = subprocess.Popen(["python", "-u", "driver.py", name, pattern],
                                   stdout=open(f"driver_{name}.log", "w"), stderr=subprocess.STDOUT)
time.sleep(4)
for name in DRIVERS:
    print(open(f"driver_{name}.log").read().strip())

## D1: publish the six routing keys from the paper table and see who received each

In [ ]:
!python send_d1_keys.py
time.sleep(2)
import re, json
for row in range(1, 7):
    receivers = []
    for name in DRIVERS:
        if re.search(rf'"d1_row": {row},', open(f"driver_{name}.log").read()):
            receivers.append(name)
    print(f"D1 row {row}: ", receivers or "dropped")

## D2 / D3: run the dispatcher for 2 minutes with all drivers running

In [ ]:
!python dispatcher.py 120 | tee dispatcher.log

In [ ]:
for name in DRIVERS:
    log = open(f"driver_{name}.log").read()
    print(f"{name}: {log.count('-> ACK')} ACKed, {log.count('-> NACK')} NACKed")
print()
print("---- Control driver log (receives every Bangalore request) ----")
print(open("driver_Control.log").read())

In [ ]:
for p in procs.values():
    p.terminate()